# Conditional Optimal Transport Lambda-Space Two-Step Multistep Method

本 notebook 从 CondOT 直线路径推导 lambda-space 二步 solver。它不抄 VPSDE DPM-Solver++ 系数；先推导 CondOT 的半线性 ODE，再对其积分形式做插值和解析积分。

$$x_t=(1-t)x_{noise}+t x_{data},\qquad v=x_{data}-x_{noise}. $$

网络输出 vθ 后，正确的数据端点预测为

$$D_\theta(x_t,t)=x_t+(1-t)v_\theta(x_t,t).$$

故

$$v_\theta(x,t)=\frac{D_\theta(x,t)-x}{1-t}. $$

## 1. 半线性结构与解析积分

代入 Flow Matching ODE `dx/dt=vθ`：

$$\frac{dx}{dt}=-\frac{x}{1-t}+\frac{D_\theta(x,t)}{1-t}. $$

第一项对 x 线性，第二项包含所有神经网络非线性。令 σ=1−t，积分因子是 1/σ：

$$\frac{d}{dt}(\frac{x}{\sigma})=\frac{D_\theta(x,t)}{\sigma^2}. $$

这并没有消除 Dθ 对 x 的非线性，但将线性漂移精确积分。冻结 D 时，一阶更新化简为 Euler，因此高阶收益必须来自对 D 的高阶插值。

### 1.1 齐次解不是被丢掉，而是成为每一步的精确状态转移

只保留线性项时有齐次方程

$$\frac{dx}{dt}=-\frac{x}{1-t}. $$

分离变量并积分可得

$$\log\lVert x(t)\rVert=\log(1-t)+C.$$

等价地，对向量状态逐分量理解为

$$x(t)=C(1-t)=C\sigma(t).$$

若在当前步的左端给定 $$x(t_n)=x_n$$，则该步的齐次常量为

$$C=\frac{x_n}{\sigma_n}. $$

因此线性项从 $$t_n$$ 到 $$t_{n+1}$$ 的精确传播不是零，而是

$$x_{n+1}^{\mathrm{linear}}=\frac{\sigma_{n+1}}{\sigma_n}x_n. $$

只有最后到达 $$t=1$$ 时，$$\sigma_{n+1}=0$$，这一个线性传播贡献才变为零。中间每一步都必须保留这个比例；它正是原始速度中 $$-x/(1-t)$$ 的精确作用。

为了处理完整方程，令已知的齐次缩放乘上一个会变化的系数：

$$x(t)=\sigma(t)y(t).$$

代入后

$$\frac{dx}{dt}=-y+\sigma\frac{dy}{dt},$$

而原方程右边为

$$\frac{D_\theta-\sigma y}{\sigma}=\frac{D_\theta}{\sigma}-y. $$

两边的 $$-y$$ 精确抵消，得到

$$\frac{dy}{dt}=\frac{D_\theta(x,t)}{\sigma^2}. $$

这不是把线性部分删掉，而是把它编码进 $$x=\sigma y$$ 和每步的 $$\sigma_{n+1}/\sigma_n$$ 精确传播中。需要数值近似的只剩神经网络 forcing $$D_\theta(x,t)$$。


### 1.2 非线性 forcing 在 t 空间怎样被计算

从变换后的方程积分并恢复 $$x=\sigma y$$：

$$x_{n+1}=\frac{\sigma_{n+1}}{\sigma_n}x_n+\sigma_{n+1}\int_{t_n}^{t_{n+1}}\frac{D_\theta(x_t,t)}{\sigma_t^2}dt. $$

decoder 在当前状态只给出一个点值。若模型预测 velocity，先换成

$$D_n=x_n+\sigma_n v_\theta(x_n,t_n).$$

二步方法保存 $$D_{n-1},D_n$$，并在当前区间把未知函数近似成线性函数：

$$D_\theta(x_t,t)\approx D_n+(t-t_n)S_n,$$

$$S_n=\frac{D_n-D_{n-1}}{t_n-t_{n-1}}. $$

此时网络不再出现在积分号中；两个已知基函数积分为

$$\int_{t_n}^{t_{n+1}}\frac{1}{\sigma_t^2}dt=\frac{1}{\sigma_{n+1}}-\frac{1}{\sigma_n},$$

$$\sigma_{n+1}\int_{t_n}^{t_{n+1}}\frac{t-t_n}{\sigma_t^2}dt=\sigma_n-\sigma_{n+1}+\sigma_{n+1}\log\frac{\sigma_{n+1}}{\sigma_n}. $$

所以 t-space CondOT 2M 更新为

$$x_{n+1}=\frac{\sigma_{n+1}}{\sigma_n}x_n+\frac{t_{n+1}-t_n}{\sigma_n}D_n+\left(\sigma_n-\sigma_{n+1}+\sigma_{n+1}\log\frac{\sigma_{n+1}}{\sigma_n}\right)S_n. $$

这解释了代码中 `d_previous`、`d_now`、`slope` 与 `integral_linear_basis` 的含义。第一步没有 $$D_{n-1}$$，所以必须一阶 warm-up；最后一步是否外推 $$S_n$$ 是稳定性选择，默认实现使用 Euler fallback，`full_endpoint` 则使用解析极限。


## 2. 为什么进入 log-SNR 坐标

CondOT 的 α=t、σ=1−t。定义

$$\lambda=\log\frac{\alpha}{\sigma}=\log\frac{t}{1-t},$$

$$\frac{dt}{d\lambda}=t(1-t)=\alpha\sigma. $$

因此

$$\frac{dx}{d\lambda}=\alpha(D_\theta-x),$$

$$\frac{d}{d\lambda}(\frac{x}{\sigma})=e^\lambda D_\theta(x,\lambda). $$

从 λn 到 λs 的精确积分形式为

$$x_s=\frac{\sigma_s}{\sigma_n}x_n+\sigma_s\int_{\lambda_n}^{\lambda_s}e^\lambda D_\theta(x_\lambda,\lambda)d\lambda. $$

真正需近似的是这个带权非线性积分。

### 2.1 从 t 积分到 lambda 积分：上下限和核如何变换

变量变换是

$$\lambda=\log t-\log(1-t).$$

因此

$$\frac{d\lambda}{dt}=\frac{1}{t}+\frac{1}{1-t}=\frac{1}{t\sigma},$$

$$\frac{dt}{d\lambda}=t\sigma. $$

链式法则给出

$$\frac{dy}{d\lambda}=\frac{dy}{dt}\frac{dt}{d\lambda}=\frac{D_\theta}{\sigma^2}(t\sigma)=\frac{t}{\sigma}D_\theta. $$

由 $$e^\lambda=t/\sigma$$，故

$$\boxed{\frac{dy}{d\lambda}=e^\lambda D_\theta(x_\lambda,\lambda).}$$

原区间 $$[t_n,t_{n+1}]$$ 的上下限逐点变换为

$$\lambda_n=\log\frac{t_n}{1-t_n},\qquad \lambda_{n+1}=\log\frac{t_{n+1}}{1-t_{n+1}}. $$

因为 $$d\lambda/dt>0$$，顺序不变。于是

$$\int_{t_n}^{t_{n+1}}\frac{D_\theta(x_t,t)}{(1-t)^2}dt=\int_{\lambda_n}^{\lambda_{n+1}}e^\lambda D_\theta(x_\lambda,\lambda)d\lambda. $$

uniform lambda 节点经 $$t=\operatorname{sigmoid}(\lambda)$$ 变回 t 空间后并不均匀。局部有

$$\Delta t\approx t(1-t)\Delta\lambda. $$

故在 $$t\to0$$ 与 $$t\to1$$ 两端，固定的 $$\Delta\lambda$$ 自动对应很小的 $$\Delta t$$。


### 2.2 为什么截断有限 lambda 区间仍可行

严格地说，开区间 $$t\in(0,1)$$ 对应

$$\lambda\in(-\infty,+\infty).$$

实际取 $$t\in[\varepsilon,1-\varepsilon]$$ 时，只计算

$$\lambda\in\left[\log\frac{\varepsilon}{1-\varepsilon},\log\frac{1-\varepsilon}{\varepsilon}\right].$$

这不是严格等价的无穷区间积分，而是端点截断加边界处理。必须区分左右两端。

左端，若 $$\lVert D_\theta\rVert\le M$$，则

$$\left\lVert\int_{-\infty}^{\lambda_{\min}}e^\lambda D_\theta d\lambda\right\rVert\le M e^{\lambda_{\min}}=M\frac{\varepsilon}{1-\varepsilon}=O(\varepsilon).$$

所以左尾通常小；当前实现还用一次 t-space bridge 近似 $$[0,\varepsilon]$$，而不是完全忽略它。

右端不能说尾积分小。若 $$D_\theta=D$$ 为常量，则

$$\int_{\lambda_{\max}}^{+\infty}e^\lambda D\,d\lambda$$

在 $$y=x/\sigma$$ 坐标中会发散。可观测的状态却是 $$x=\sigma y$$，并有边界关系

$$\lim_{t\to1}x_t=D. $$

因此 solver 在 $$t=1-\varepsilon$$ 评估 $$D_\theta(x_{1-\varepsilon},1-\varepsilon)$$ 后，返回它作为 $$x_1$$ 的 terminal projection。对冻结的常量 D，这恰好是最后一段 $$[1-\varepsilon,1]$$ 的解析解，而不是丢弃右尾。

$$\varepsilon$$ 是截断偏差和端点数值稳定性的折中：过小会扩大有限 lambda 区间、增加极端时间处网络误差与转换病态；过大则增加 bridge 和 terminal projection 的偏差。score 参数化在左端尤其敏感，因为 score-to-velocity 转换含有 $$1/t$$。


### 2.3 两个截断小区间如何实际计算

lambda solver 不是把两个端点区间直接删除，而是将整个路径拆为

$$[0,1]=[0,\varepsilon]+[\varepsilon,1-\varepsilon]+[1-\varepsilon,1].$$

左端 $$[0,\varepsilon]$$：lambda 在 $$t=0$$ 无定义，当前实现用一次左端 bridge

$$x_{\varepsilon}\approx x_0+\varepsilon v_\theta(x_0,0).$$

这是一阶近似而非严格删除。若速度有界，遗漏的高阶项随 $$\varepsilon$$ 缩小。score 参数化在这里最敏感，因为 score-to-velocity 含有 $$1/t$$；更严格的 score sampler 应设计从有限 $$t=\varepsilon$$ 开始的初始化。

右端 $$[1-\varepsilon,1]$$：这段不能因为 lambda 尾积分看起来麻烦就忽略。设

$$t_a=1-\varepsilon,\qquad x_a=x_{t_a},\qquad D_a=D_\theta(x_a,t_a).$$

对网络 forcing 做左端零阶保持：

$$D_\theta(x_t,t)\approx D_a,\qquad t\in[t_a,1].$$

注意冻结的是 $$D$$，不是整个速度。线性项仍按解析解处理。令

$$z_t=D_a-x_t.$$

则原 ODE 给出

$$\frac{dz_t}{dt}=-\frac{z_t}{1-t}. $$

以 $$z_{t_a}=D_a-x_a$$ 为初值，其解析解为

$$z_t=\frac{1-t}{\varepsilon}(D_a-x_a),$$

$$x_t=D_a-\frac{1-t}{\varepsilon}(D_a-x_a).$$

因此在终点严格得到

$$\boxed{x_1=D_a.}$$

所以代码在最后有限 lambda 节点计算 $$D_a$$ 后直接 `return D_a`。这叫 terminal projection，是最后短区间在冻结 D 假设下的解析解，不是忽略右尾。

对 CondOT，此解析终点更新恰好与左端 Euler 代数相同。因为 $$h=1-t_a=\varepsilon$$：

$$x_1=x_a+h\frac{D_a-x_a}{\varepsilon}=D_a. $$

二者结果相同是这个特定线性系数的巧合；推导和实现应理解为“冻结 D + 精确线性传播”。


## 3. 二步线性插值的严格系数

用两个历史 data prediction 在 λ 上线性插值：

$$D(\lambda)\approx D_n+\frac{D_n-D_{n-1}}{h_0}(\lambda-\lambda_n),$$

$$h=\lambda_s-\lambda_n,\qquad h_0=\lambda_n-\lambda_{n-1}. $$

两个积分是

$$I_0=e^{\lambda_s}-e^{\lambda_n},$$

$$I_1=e^{\lambda_s}(h-1)+e^{\lambda_n}. $$

所以严格线性插值更新是

$$x_s=\frac{\sigma_s}{\sigma_n}x_n+\sigma_s I_0D_n+\sigma_s\frac{I_1}{h_0}(D_n-D_{n-1}).$$

代码的 `exact_linear` 正是此式。`dpmpp` 选项则采用 Diffusion-Planner 风格的二阶近似系数，以便公平消融。端点 λ=∞ 不能插值，因此从 ε 走到 1−ε，并返回已经评估的 D(x₁₋ε,1−ε)。

In [ ]:
import torch


class CondOTLambdaTwoStep:
    """Self-contained lambda-space CondOT two-step sampler.

    ``velocity_model(x, t)`` returns the raw Flow-Matching velocity.  Each
    decoder evaluation is converted once into D=x+(1-t)v.  The linear drift
    -x/(1-t) is integrated analytically; only D(lambda) is interpolated.
    """

    def __init__(self, velocity_model, variant="exact_linear"):
        if variant not in {"exact_linear", "dpmpp"}:
            raise ValueError("variant must be 'exact_linear' or 'dpmpp'")
        self.velocity_model = velocity_model
        self.variant = variant
        self.nfe = 0

    @staticmethod
    def lam(t):
        """CondOT log-SNR coordinate lambda=log(t/(1-t))."""
        return torch.log(t) - torch.log1p(-t)

    @torch.no_grad()
    def sample(self, x_init, nfe=8, eps=1e-3):
        if nfe < 2:
            raise ValueError("two-step lambda sampling requires nfe >= 2")
        if not 0.0 < eps < 0.5:
            raise ValueError("eps must lie in (0, 0.5)")

        eps_t = torch.tensor(eps, device=x_init.device, dtype=x_init.dtype)
        lambda_grid = torch.linspace(
            self.lam(eps_t), self.lam(1.0 - eps_t), nfe - 1,
            device=x_init.device, dtype=x_init.dtype,
        )
        time_grid = torch.sigmoid(lambda_grid)
        x = x_init
        self.nfe = 0

        # lambda(0) is -infinity: bridge [0, eps] once in physical t-space.
        v0 = self.velocity_model(x, torch.zeros((), device=x.device, dtype=x.dtype))
        self.nfe += 1
        x = x + eps_t * v0

        d_previous = None
        lambda_previous = None
        for index, (t_now, lambda_now) in enumerate(zip(time_grid, lambda_grid)):
            v_now = self.velocity_model(x, t_now)
            self.nfe += 1
            d_now = x + (1.0 - t_now) * v_now

            # Freezing D on [1-eps, 1] has exact terminal state x_1=D_now.
            if index == len(time_grid) - 1:
                return d_now

            t_next, lambda_next = time_grid[index + 1], lambda_grid[index + 1]
            h = lambda_next - lambda_now
            sigma_now, sigma_next = 1.0 - t_now, 1.0 - t_next
            # sigma_next * integral exp(lambda) d lambda for a constant D.
            coefficient_0 = -t_next * torch.expm1(-h)

            if index == 0:
                # First interior interval has no D_{n-1}; use zero-order hold.
                x = (sigma_next / sigma_now) * x + coefficient_0 * d_now
            else:
                h0 = lambda_now - lambda_previous
                delta_d = d_now - d_previous
                if self.variant == "exact_linear":
                    # Exact integral when D(lambda) is linear through two history points.
                    coefficient_1 = t_next * (torch.expm1(-h) + h) / h0
                else:
                    # DPM-Solver++-style second-order coefficient, used as an ablation.
                    coefficient_1 = 0.5 * coefficient_0 * h / h0
                x = ((sigma_next / sigma_now) * x
                     + coefficient_0 * d_now
                     + coefficient_1 * delta_d)

            d_previous, lambda_previous = d_now, lambda_now


# Constant velocity v=2 from x(0)=0 has endpoint x(1)=2 exactly.
for variant in ("exact_linear", "dpmpp"):
    solver = CondOTLambdaTwoStep(lambda x, t: torch.full_like(x, 2.0), variant)
    output = solver.sample(torch.zeros(2, 2), nfe=8)
    torch.testing.assert_close(output, torch.full_like(output, 2.0), atol=2e-5, rtol=0)
    assert solver.nfe == 8
print("both lambda-space variants passed their constant-velocity and NFE checks")



## 4. 代码审计：公式、NFE 与实现边界

独立代码单元和 `solvers/condot_2m.py` 使用相同的计算顺序：

1. 左端 $t=0$ 无有限 $\lambda$，所以先调用一次速度并做 $[0,\varepsilon]$ Euler bridge；
2. 取 $nfe-1$ 个从 $\lambda(\varepsilon)$ 到 $\lambda(1-\varepsilon)$ 的节点；每个节点只调用一次 decoder，并转成 $D_n=x_n+(1-t_n)v_n$；
3. 第一个内部区间冻结 $D$，以后用 $D_{n-1},D_n$ 做二步更新；
4. 最后节点已经得到 $D_{1-\varepsilon}$，以 terminal projection 返回它。

因此总调用数恰为

$$
1+(nfe-1)=nfe.
$$

`exact_linear` 的二项系数是线性插值积分的精确值；`dpmpp` 的二项系数是 DPM-Solver++ 风格的二阶近似，二者不是同一公式。后者只用于消融比较，不能因为名字相似就假设它在 CondOT 上必然优于 AB2。

还应区分两种 `CondOT 2M`：仓库中的 `CondOT2MSolver` 在 **t 空间**运行，并可以用 robust Euler final fallback；本笔记的 `CondOTLambdaTwoStep` 在 **lambda 空间**运行，采用 bridge + terminal projection。它们都利用半线性结构，但时间节点、末端处理和误差常数不同，实验结果不必相同。
